# Libraries and Data Import

We use data downloaded from OpenStreetMap (OSM) in PBF format:

https://download.geofabrik.de/europe/spain-latest.osm.pbf

In [1]:
import json
import osmium
import pandas as pd

# Data Analysis and Cleaning

In [17]:
class ServicioHandler(osmium.SimpleHandler):
    def __init__(self):
        super(ServicioHandler, self).__init__()
        self.data = []

        # Mapping from OpenStreetMaps to the system
        # Structure: 'Key_OSM': { 'Value_OSM': 'Enum_TFG' }
        self.mapping = {
            'amenity': {
                'cafe': 'cafe',
                'restaurant': 'restaurant',
                'fast_food': 'restaurant',
                'pharmacy': 'pharmacy',
                'atm': 'atm'
            },
            'shop': {
                'supermarket': 'supermarket',
                'convenience': 'supermarket',
                'car_repair': 'mechanic'
            },
            'tourism': {
                'motel': 'motel',
                'hotel': 'motel',
                'hostel': 'motel',
                'guest_house': 'motel'
            },
            'craft': {
                'car_mechanic': 'mechanic'
            }
        }

    def node(self, n):
        types = set()

        for tag in n.tags:
            if tag.k in self.mapping:
                # The key of the tag is in our dict
                if tag.v in self.mapping[tag.k]:
                    # The value of the tag is in the sub-dict of the key
                    types.add(self.mapping[tag.k][tag.v])

        for t in types:
            self.data.append({
                'name': n.tags.get('name', 'Unknown')[:100],
                'type': t,
                'location': (n.location.lat, n.location.lon)
            })

In [18]:
handler = ServicioHandler()
handler.apply_file('spain-260210.osm.pbf')

In [22]:
data = handler.data
data_df = pd.DataFrame(data)
data_df

,name,type,location
0,Exe Pozuelo,motel,"(40.3984408, -3.7881758)"
1,Café Comercial,restaurant,"(40.4287093, -3.701972)"
2,Honest Greens,restaurant,"(40.4270276, -3.7016997)"
3,Alcampo,supermarket,"(40.480352, -3.7068876)"
4,Burger King,restaurant,"(40.4333661, -3.6072228)"
...,...,...,...
141506,Estanco-Panadería-Prensa,supermarket,"(39.9426006, -4.4381243)"
141507,Farmacia Gómez Cañizares,pharmacy,"(39.9438436, -4.4369364)"
141508,El Super de Pilar,supermarket,"(39.9430787, -4.4373146)"
141509,Casa Rural Camino Real,motel,"(39.9404315, -4.4388231)"


In [24]:
print(data_df["type"].value_counts().sort_values(ascending=False))

type
restaurant     60424
supermarket    21818
cafe           20673
pharmacy       17267
motel          12137
mechanic        5547
atm             3645
Name: count, dtype: int64


In [25]:
with open('services_data.json', 'w', encoding='utf-8') as f:
    json.dump(data, f, indent=4, ensure_ascii=False)

# DataBase Population

In [26]:
import sys
sys.path.append('../..')

from src.Persistance.db_broker import DBBroker

In [27]:
with open('services_data.json', 'r') as f:
    clean_data = json.load(f)

len(clean_data)

141511

In [28]:
clean_data_df = pd.DataFrame(clean_data)
clean_data_df

,name,type,location
0,Exe Pozuelo,motel,"[40.3984408, -3.7881758]"
1,Café Comercial,restaurant,"[40.4287093, -3.701972]"
2,Honest Greens,restaurant,"[40.4270276, -3.7016997]"
3,Alcampo,supermarket,"[40.480352, -3.7068876]"
4,Burger King,restaurant,"[40.4333661, -3.6072228]"
...,...,...,...
141506,Estanco-Panadería-Prensa,supermarket,"[39.9426006, -4.4381243]"
141507,Farmacia Gómez Cañizares,pharmacy,"[39.9438436, -4.4369364]"
141508,El Super de Pilar,supermarket,"[39.9430787, -4.4373146]"
141509,Casa Rural Camino Real,motel,"[39.9404315, -4.4388231]"


In [32]:
num_unknown = (clean_data_df["name"] == "Unknown").sum()
print(f'Services named "Unknown": {num_unknown}')

Services named "Unknown": 14652


In [33]:
query = """
    INSERT INTO services (service_name, service_type, location)
    VALUES (?, ?, ST_GeomFromText(?))
"""

In [35]:
batch_size = 10000

for i in range(0, len(clean_data), batch_size):
    batch = [
        (item['name'], item['type'], f"POINT({item['location'][1]} {item['location'][0]})")
        for item in clean_data[i : i + batch_size]
    ]
    DBBroker().execute_many(query, batch)

2026-03-05 19:46:48,065 - DBBroker - DEBUG - Connection used from pool.
2026-03-05 19:46:48,065 - DBBroker - DEBUG - Executing batch WRITE query: 
    INSERT INTO services (service_name, service_type, location)
    VALUES (?, ?, ST_GeomFromText(?))
 | Batch size: 10000
2026-03-05 19:46:48,255 - DBBroker - DEBUG - Batch write operation successful. Total rows affected: 0
2026-03-05 19:46:48,256 - DBBroker - DEBUG - Connection returned to pool.
2026-03-05 19:46:48,262 - DBBroker - DEBUG - Connection used from pool.
2026-03-05 19:46:48,263 - DBBroker - DEBUG - Executing batch WRITE query: 
    INSERT INTO services (service_name, service_type, location)
    VALUES (?, ?, ST_GeomFromText(?))
 | Batch size: 10000
2026-03-05 19:46:48,350 - DBBroker - DEBUG - Batch write operation successful. Total rows affected: 0
2026-03-05 19:46:48,351 - DBBroker - DEBUG - Connection returned to pool.
2026-03-05 19:46:48,357 - DBBroker - DEBUG - Connection used from pool.
2026-03-05 19:46:48,358 - DBBroker -